# Instruction Fine-Tuning: TinyGPT on Alpaca Cleaned (52K)

**Goal:** Take the pretrained TinyGPT base model and instruction fine-tune it
using the Alpaca Cleaned dataset (52K examples), using a custom PyTorch training loop.

**Setup required before running on Kaggle:**
1. Turn on Internet in the Kaggle notebook settings
2. Select a GPU accelerator
3. Run Step 1 to clone the TinyGPT repo and download `tinygpt_weights.pt` from Hugging Face

## Step 1: Kaggle Setup and Download Base Model

In [ ]:
!pip install -q tiktoken datasets

# Kaggle working paths
!mkdir -p /kaggle/working/models

# Clone the TinyGPT code if this notebook is running by itself on Kaggle.
!test -f /kaggle/working/tinygpt/tinygpt.py || git clone https://github.com/hemantvirmani/tinygpt /kaggle/working/tinygpt

# Download the public PyTorch-native TinyGPT weights from Hugging Face.
# This is the format used by this fine-tuning notebook.
!wget -q --show-progress -O /kaggle/working/models/tinygpt_weights.pt https://huggingface.co/hemantvirmani/tinyGPT/resolve/main/tinygpt_weights.pt

!ls -lh /kaggle/working/models/tinygpt_weights.pt

## Step 2: Config

In [ ]:
import sys
import os
import inspect
import torch
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
from datasets import load_dataset

# Match the pretraining script's matmul precision setting.
torch.set_float32_matmul_precision("high")

# --- Paths ---
TINYGPT_DIR     = "/kaggle/working/tinygpt"              # where tinygpt.py lives
MODEL_DIR       = "/kaggle/working/models"
WEIGHTS_PATH    = f"{MODEL_DIR}/tinygpt_weights.pt"
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Fine-tuning Hyperparameters ---
BATCH_SIZE           = 4
EFFECTIVE_BATCH_SIZE = 64    # accumulation_steps = 64/4 = 16
MAX_STEPS            = 3000
LEARNING_RATE        = 1e-4  # lower than pretraining
WEIGHT_DECAY         = 0.01
GRAD_CLIP            = 1.0
WARMUP_STEPS         = 100
EVAL_STEPS           = 100   # evaluate every 100 steps
EVAL_ITERATIONS      = 20    # batches averaged during validation
SAVE_STEPS           = 500   # save checkpoint every N steps
MAX_SEQ_LEN          = 512   # shorter than pretraining 1024 — Alpaca examples are short
VAL_SPLIT            = 0.1
# BF16 requires Ampere (A100, RTX 3090+). Kaggle T4/P100 don't support it — auto-detect.
USE_BF16             = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"BF16: {USE_BF16}")

## Step 3: Load TinyGPT Model

In [ ]:
sys.path.insert(0, TINYGPT_DIR)
import tinygpt

# Load tokenizer
enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # 50256 — used as padding and end-of-text

# Load model in TRAIN mode (not eval)
state = tinygpt.State(
    tokenizer=enc,
    train_data=None,
    val_data=None,
    vocab_size=enc.n_vocab
)
model = tinygpt.TinyGPT(state).to(device)

# Load pretrained weights (PyTorch native format)
state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
if any(k.startswith("_orig_mod.") for k in state_dict):
    state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.train()  # switch to training mode

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Parameters: {total_params/1e6:.1f}M")

## Step 4: Load and Format Alpaca Cleaned

The core of instruction fine-tuning — we define a prompt template that teaches
the model what an instruction looks like, what a response looks like, and where
it ends. The model learns this structure through repeated exposure.

In [ ]:
print("Loading Alpaca Cleaned (52K)...")
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f"Total examples: {len(dataset)}")
print(f"Sample: {dataset[0]}")

In [ ]:
def format_and_tokenize(example):
    """Format an Alpaca example into instruction-response format and tokenize.

    Returns a list of token ids, or None if the example is too long.
    The EOT token marks the end of the response — critical for the model
    to learn where to stop generating.
    """
    instruction = example["instruction"].strip()
    input_text  = example["input"].strip()   # optional context (called "input" in Alpaca)
    response    = example["output"].strip()  # called "output" in Alpaca

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{response}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        )

    tokens = enc.encode_ordinary(text)
    tokens.append(EOT)  # end of sequence marker

    if len(tokens) > MAX_SEQ_LEN:
        return None  # skip examples that are too long
    return tokens


print("Tokenizing dataset...")
all_tokens = []
skipped = 0
for ex in dataset:
    tokens = format_and_tokenize(ex)
    if tokens is None:
        skipped += 1
        continue
    all_tokens.append(tokens)

print(f"Kept: {len(all_tokens)} examples | Skipped (too long): {skipped}")

# Train / val split
split_idx  = int(len(all_tokens) * (1 - VAL_SPLIT))
train_data = all_tokens[:split_idx]
val_data   = all_tokens[split_idx:]
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# Preview formatted example
print("\nFormatted example (decoded):")
print(enc.decode(train_data[0]))

## Step 5: Batch Sampler

Since Alpaca examples have variable lengths, we pad shorter sequences to
the longest in the batch. Padding tokens are masked out in the loss
so they don't affect training.

In [ ]:
import random

def _get_batch(split):
    """Sample a random Alpaca batch, pad it, and return (x, y, mask)."""
    data = train_data if split == "train" else val_data
    batch = random.sample(data, BATCH_SIZE)

    max_len = max(len(s) for s in batch)

    x_list, y_list, mask_list = [], [], []
    for tokens in batch:
        pad_len = max_len - len(tokens)
        # x: input tokens (all but last)
        # y: target tokens (all but first, shifted by 1)
        padded = tokens + [EOT] * pad_len
        x = padded[:-1]
        y = padded[1:]
        # mask: 1 where tokens are real, 0 where padded
        mask = [1] * (len(tokens) - 1) + [0] * pad_len
        x_list.append(x)
        y_list.append(y)
        mask_list.append(mask)

    x    = torch.tensor(x_list, dtype=torch.long).to(device)
    y    = torch.tensor(y_list, dtype=torch.long).to(device)
    mask = torch.tensor(mask_list, dtype=torch.float).to(device)
    return x, y, mask


def _compute_loss(model, x, y, mask):
    """Forward pass + masked cross entropy loss."""
    logits = model(x)                          # (B, T, vocab)
    B, T, C = logits.shape
    loss = F.cross_entropy(
        logits.view(B * T, C),
        y.view(B * T),
        reduction="none"
    )                                          # (B*T,)
    loss = loss * mask.view(B * T)             # zero out padding positions
    return loss.sum() / mask.sum()             # mean over real tokens only


@torch.no_grad()
def _evaluate_loss(model, eval_iters=EVAL_ITERATIONS):
    """Averages loss over multiple batches for a more stable validation metric."""
    was_training = model.training
    model.eval()
    losses = torch.zeros(eval_iters, device=device)
    for k in range(eval_iters):
        val_x, val_y, val_mask = _get_batch(split="val")
        if device == "cuda" and USE_BF16:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                losses[k] = _compute_loss(model, val_x, val_y, val_mask)
        else:
            losses[k] = _compute_loss(model, val_x, val_y, val_mask)
    val_loss = losses.mean()
    if was_training:
        model.train()
    return val_loss

print("Batch sampler ready.")

## Step 6: Baseline — Test BEFORE Fine-Tuning

In [ ]:
test_prompts = [
    "What is photosynthesis?",
    "Explain the water cycle in simple terms.",
    "Who was the first emperor of Rome?",
]

def test_model(model, prompts, label=""):
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    model.eval()
    for prompt in prompts:
        full_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"
        print(f"\nQ: {prompt}")
        print(f"A: {model.generate_text(start_text=full_prompt, max_tokens=150, temperature=0.7)}")
        print("-" * 40)
    model.train()

test_model(model, test_prompts, label="BASELINE (Before Fine-Tuning)")

## Step 7: Optimizer and Scheduler

In [ ]:
def _configure_optimizers(model, weight_decay, learning_rate, device_type):
    # start with all of the candidate parameters (that require grad)
    param_dict = {pn: p for pn, p in model.named_parameters()}
    param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
    # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
    print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
    # Create AdamW optimizer and use the fused version if it is available
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device_type == "cuda"
    print(f"using fused AdamW: {use_fused}")
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
    return optimizer


def _setup_training(model):
    
    optimizer = _configure_optimizers(
        model,
        weight_decay=WEIGHT_DECAY,
        learning_rate=LEARNING_RATE,
        device_type=device)

    # 1. Define the Warmup Scheduler (Linear increase from a 10% of LEARNING_RATE to LEARNING_RATE)
    scheduler_warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1,end_factor=1.0,
        total_iters=WARMUP_STEPS)

    # 2. Define the Main Scheduler (Cosine decay from LEARNING_RATE to eta_min)
    scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=(MAX_STEPS - WARMUP_STEPS),
        eta_min=0.1*LEARNING_RATE) # Decay to 10% of LEARNING_RATE by the end of training

    # 3. Combine them using SequentialLR - milestones=[WARMUP_STEPS] means switch to the second scheduler at step WARMUP_STEPS
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[scheduler_warmup, scheduler_cosine],
        milestones=[WARMUP_STEPS]
    )

    return optimizer, scheduler

## Step 8: Resume from Checkpoint (Optional)

If resuming a previous fine-tuning run, set `RESUME_CKPT` to the checkpoint
file path and run this cell. Skip if starting fresh.

In [ ]:
# Set to None to start fresh, or path to resume e.g.:
# RESUME_CKPT = f"{MODEL_DIR}/tinygpt_alpaca_step1000.pt"
RESUME_CKPT = None


def _maybe_load_checkpoint(model, optimizer=None, scheduler=None, resume_path=RESUME_CKPT):
    if not resume_path:
        print("Starting fresh from step 0.")
        return 0

    if not os.path.exists(resume_path):
        print(f"Checkpoint not found at {resume_path}. Starting fresh.")
        return 0

    ckpt = torch.load(resume_path, map_location=device, weights_only=False)
    # Support both full training checkpoints and bare inference-only state dicts.
    if isinstance(ckpt, dict) and "model_state" in ckpt:
        state_dict = ckpt["model_state"]
        if optimizer is not None:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        if scheduler is not None and ckpt.get("scheduler_state") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        start_step = int(ckpt.get("step", 0)) + 1
        print(f"Resumed from {resume_path} at step {start_step}")
    else:
        state_dict = ckpt
        start_step = 0
        print(f"Loaded weights from {resume_path}")

    if any(k.startswith("_orig_mod.") for k in state_dict):
        state_dict = {k.removeprefix("_orig_mod."): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    return start_step


def _maybe_save_checkpoint(model, optimizer, scheduler=None, step=0, vocab_size=0):
    if (step + 1) % SAVE_STEPS != 0:
        return

    resume_path = f"{MODEL_DIR}/tinygpt_alpaca_step{step+1}.pt"
    payload = {
        "step": step,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "vocab_size": vocab_size,
    }
    if scheduler is not None:
        try:
            payload["scheduler_state"] = scheduler.state_dict()
        except Exception:
            payload["scheduler_state"] = None
    torch.save(payload, resume_path)
    print(f"Saved checkpoint: {resume_path}")


def _plot_losses(steps, train_losses, val_losses):
    if not steps:
        return

    output_path = f"{MODEL_DIR}/finetune_loss_curve.png"
    plt.figure(figsize=(10, 6))
    plt.plot(steps, train_losses, label="train")
    plt.plot(steps, val_losses, label="val")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Fine-Tuning Loss — TinyGPT on Alpaca Cleaned")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()
    print(f"Loss curve saved to {output_path}")

## Step 9: Training Loop

In [ ]:
def train_loop(model):
    optimizer, scheduler = _setup_training(model)

    # Target an effective batch size without exceeding GPU memory.
    # Since BATCH_SIZE is small, we accumulate gradients over multiple micro-batches.
    assert EFFECTIVE_BATCH_SIZE % BATCH_SIZE == 0, "EFFECTIVE_BATCH_SIZE must be divisible by BATCH_SIZE"
    accumulation_steps = EFFECTIVE_BATCH_SIZE // BATCH_SIZE

    print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE} (via {accumulation_steps} accumulation steps)")

    start_step = _maybe_load_checkpoint(model, optimizer, scheduler)

    steps = []
    train_losses = []
    val_losses = []

    for step in range(start_step, MAX_STEPS):
        # 1. Accumulate gradients over multiple micro-batches
        optimizer.zero_grad(set_to_none=True)
        micro_step_loss = 0.0

        for _ in range(accumulation_steps):
            x, y, mask = _get_batch(split="train")
            if device == "cuda" and USE_BF16:
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    loss = _compute_loss(model, x, y, mask)
            else:
                loss = _compute_loss(model, x, y, mask)
            # Scale loss by accumulation steps so gradients are averaged correctly
            scaled_loss = loss / accumulation_steps
            scaled_loss.backward()
            micro_step_loss += loss.item()

        avg_train_loss = micro_step_loss / accumulation_steps

        # 2. Clip and update weights
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        # 3. Logging and Checkpointing
        if (step + 1) % EVAL_STEPS == 0 or step == start_step:
            val_loss = _evaluate_loss(model, eval_iters=EVAL_ITERATIONS)
            steps.append(step + 1)
            train_losses.append(avg_train_loss)
            val_losses.append(val_loss.item())
            print(f"step {step+1}: train {avg_train_loss:.4f} | val {val_loss.item():.4f}")

        _maybe_save_checkpoint(model=model, optimizer=optimizer, scheduler=scheduler,
            vocab_size=enc.n_vocab, step=step)

    _plot_losses(steps, train_losses, val_losses)
    return steps, train_losses, val_losses


steps, train_losses, val_losses = train_loop(model)
print("\nTraining complete!")

## Step 10: Plot Loss Curves

In [ ]:
# The training loop already calls this once at the end.
# Re-run this cell if you want to regenerate the plot.
_plot_losses(steps, train_losses, val_losses)

## Step 11: Test AFTER Fine-Tuning

Same prompts as baseline. Compare the delta — that's your intuition moment.

In [ ]:
test_model(model, test_prompts, label="AFTER Instruction Fine-Tuning")

## Step 12: Save Final Weights

In [ ]:
# Save inference-only weights (no optimizer state)
final_weights_path = f"{MODEL_DIR}/tinygpt_alpaca_weights.pt"
state_dict = model.state_dict()
torch.save(state_dict, final_weights_path)

size_mb = os.path.getsize(final_weights_path) / 1024 / 1024
print(f"Final weights saved: {final_weights_path} ({size_mb:.0f} MB)")

## What Just Happened?

| Stage | What the model learned |
|-------|------------------------|
| Pretraining (base model) | Predict next tokens from raw FineWeb-Edu text |
| Instruction fine-tuning (this notebook) | Recognize `### Instruction / ### Response` structure and generate helpful answers |

**Dataset:** `yahma/alpaca-cleaned` — 52K instruction/response pairs, cleaned version of Stanford Alpaca.

**Next step:** Take this instruction-tuned TinyGPT and do domain SFT on
Indian mythology Q&A pairs — or repeat this pipeline on Qwen3-4B with LoRA
for a production-grade result.